# N=2 Homodyne eta_minus scan with final-state fidelity histograms
Notebook version aligned with the fixed script:
- no `best_state` ranking
- per-trajectory fidelities on fixed state library
- histogram plots of fidelity distributions for each `eta_minus`


In [1]:
import os
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")

import importlib
import quantum_function2
importlib.reload(quantum_function2)

import json
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from qutip import fidelity, ket2dm, qsave, smesolve

from quantum_function2 import (
    H_free,
    PlusPlus,
    Sz_1,
    Sz_2,
    angle_to_path,
    angle_to_tex,
    concurrence_for_solver_general,
    css_2,
    ee,
    eg,
    ge,
    gg,
    phi_minus,
    phi_plus,
    psi_minus,
    psi_plus,
    save_ineff_df_npz,
    xi_KU_solver,
)


In [2]:
# --- SAFETY SWITCH ---
RUN_THIS_CELL = True
if not RUN_THIS_CELL:
    raise SystemExit("Cell locked. Set RUN_THIS_CELL=True to run")

USE_TEX = True
plt.rcParams["text.usetex"] = USE_TEX
plt.rcParams.update({
    "mathtext.fontset": "cm",
    "font.family": "serif",
    "font.size": 14,
    "axes.unicode_minus": False,
})


In [3]:
# DEFINITIONS (run once)
DEFAULT_EOPS = [
    ("Conc", concurrence_for_solver_general),
    ("Xi2_KU", xi_KU_solver),
]

LABELS = {
    "Conc": r"$\overline{\mathcal{C}}$",
    "Xi2_KU": r"$\overline{\xi^2_{KU}}$",
}


def fmt_eta(x: float) -> str:
    return f"{x:.3g}"


def build_initial_state(kind: str, theta: float, phi_state: float):
    if kind == "plusplus":
        return PlusPlus
    if kind == "ee":
        return ee
    if kind == "css":
        return css_2(theta, phi_state)
    raise ValueError(f"Unknown state kind: {kind}")


def build_homodyne_ops_eta_minus(gamma: float, phi1: float, phi2: float, eta_minus: float):
    if eta_minus < 0.0 or eta_minus > 1.0:
        raise ValueError(f"eta_minus must be in [0,1], got {eta_minus}")

    l_plus = np.sqrt(gamma / 2.0) * np.exp(1j * phi1) * (Sz_1 + Sz_2)
    l_minus = np.sqrt(gamma / 2.0) * np.exp(1j * phi2) * (Sz_1 - Sz_2)

    sc_ops = [l_plus]
    if eta_minus > 0.0:
        sc_ops.append(np.sqrt(eta_minus) * l_minus)

    c_ops = []
    if eta_minus < 1.0:
        c_ops.append(np.sqrt(1.0 - eta_minus) * l_minus)

    return c_ops, sc_ops


def _chunk_mean_expect(expect_chunk: np.ndarray, n_eops: int, n_times: int):
    arr = np.asarray(expect_chunk)
    arr = np.real_if_close(arr)
    arr = np.asarray(arr, dtype=float)

    if arr.ndim == 3 and arr.shape[0] == n_eops and arr.shape[2] == n_times:
        return arr.mean(axis=1)

    if arr.ndim == 2 and arr.shape == (n_eops, n_times):
        return arr

    if n_eops == 1 and arr.ndim == 1 and arr.shape[0] == n_times:
        return arr.reshape(1, -1)

    raise ValueError(f"Unexpected expect shape {arr.shape}")


def run_homodyne_avg_eta_minus_with_finals(
    rho0,
    times: np.ndarray,
    gamma: float,
    phi1: float,
    phi2: float,
    eta_minus: float,
    e_ops: list,
    ntraj: int,
    chunk_size: int,
    num_cpus: int,
    seed: int,
):
    options = {
        "keep_runs_results": True,
        "store_states": False,
        "store_final_state": True,
        "num_cpus": max(1, num_cpus - 1),
        "map": "parallel" if num_cpus > 1 else "serial",
    }

    n_eops = len(e_ops)
    n_times = len(times)
    weighted = None
    done = 0
    chunk_id = 0
    rng = np.random.default_rng(seed)

    final_states_all = []

    while done < ntraj:
        chunk_id += 1
        n_this = min(chunk_size, ntraj - done)
        np.random.seed(int(rng.integers(0, 2**31 - 1)))

        c_ops, sc_ops = build_homodyne_ops_eta_minus(
            gamma=gamma,
            phi1=phi1,
            phi2=phi2,
            eta_minus=eta_minus,
        )

        sol = smesolve(
            H_free,
            rho0,
            times,
            c_ops=c_ops,
            sc_ops=sc_ops,
            heterodyne=False,
            e_ops=e_ops,
            ntraj=n_this,
            options=options,
        )

        mean_chunk = _chunk_mean_expect(sol.expect, n_eops=n_eops, n_times=n_times)

        if weighted is None:
            weighted = mean_chunk * n_this
        else:
            weighted += mean_chunk * n_this

        if hasattr(sol, "runs_final_states") and sol.runs_final_states is not None:
            final_states_all.extend(sol.runs_final_states)
        elif hasattr(sol, "runs_states") and sol.runs_states is not None:
            final_states_all.extend([traj[-1] for traj in sol.runs_states])

        done += n_this
        print(f"[eta_minus={eta_minus:.3f}] chunk {chunk_id}: {done}/{ntraj} trajectories completed")

    mean_expect = weighted / float(ntraj)
    return mean_expect, final_states_all


def build_candidate_library():
    return {
        "phi_plus": phi_plus,
        "phi_minus": phi_minus,
        "psi_plus": psi_plus,
        "psi_minus": psi_minus,
        "plusplus": PlusPlus,
        "ee": ee,
        "eg": eg,
        "ge": ge,
        "gg": gg,
    }


def fidelity_table(final_states, library: dict):
    labels = list(library.keys())
    if len(final_states) == 0:
        cols = ["traj"] + [f"F_{lab}" for lab in labels]
        return pd.DataFrame(columns=cols)

    rows = []
    for i, rho_f in enumerate(final_states, start=1):
        row = {"traj": i}
        for lab in labels:
            row[f"F_{lab}"] = float(np.real_if_close(fidelity(rho_f, library[lab])))
        rows.append(row)
    return pd.DataFrame(rows)


def average_final_state(final_states):
    if len(final_states) == 0:
        return None
    rho = 0 * final_states[0]
    for r in final_states:
        rho = rho + r
    return rho / len(final_states)


def plot_fidelity_histograms(ftab: pd.DataFrame, state_labels: list[str], eta_minus: float, out_path: Path):
    n = len(state_labels)
    ncols = 3
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(5.0 * ncols, 3.6 * nrows), sharex=True)
    axes = np.asarray(axes).reshape(-1)

    for i, lab in enumerate(state_labels):
        ax = axes[i]
        col = f"F_{lab}"
        vals = ftab[col].to_numpy(dtype=float) if col in ftab.columns else np.array([], dtype=float)
        if vals.size > 0:
            ax.hist(vals, bins=np.linspace(0.0, 1.0, 26), color="#4472C4", alpha=0.85, edgecolor="white")
        ax.set_xlim(0.0, 1.0)
        ax.set_title(lab)
        ax.grid(True, linestyle=":", alpha=0.5)
        ax.set_ylabel("count")
        ax.set_xlabel("fidelity")

    for j in range(n, len(axes)):
        axes[j].axis("off")

    fig.suptitle(rf"Final-state fidelity histograms, $\eta_-={eta_minus:.2f}$", y=1.01)
    fig.tight_layout()
    fig.savefig(out_path, bbox_inches="tight")
    plt.close(fig)


In [4]:
# PARAMETERS (edit here)
state_kind = "plusplus"   # "plusplus", "ee", "css"
theta = np.pi / 2.0        # used only if state_kind == "css"
phi_state = 0.0            # used only if state_kind == "css"

gamma = 1.0
phi1 = 0.0
phi2 = 0.0

eta_plus = 1.0
eta_minus_list = np.linspace(0.0, 1.0, 5)   # 5 points, extremes included

ntraj = 30
chunk_size = 10
num_cpus = max(1, os.cpu_count() - 1)
seed = 12345

T1 = 1.0 / gamma
t_end = 10.0
dt = 0.01
times = np.arange(0.0, t_end * T1, dt * T1)

out_root = Path(r".\Graphs\Jz_2_Homodyne_N2_eta_minus_scan_final_state")
tag = ""

psi0 = build_initial_state(state_kind, theta, phi_state)
rho0 = ket2dm(psi0)

columns = [name for name, _ in DEFAULT_EOPS]
e_ops = [op for _, op in DEFAULT_EOPS]

print(f"eta_plus fixed = {eta_plus}")
print("eta_minus_list =", eta_minus_list)


eta_plus fixed = 1.0
eta_minus_list = [0.   0.25 0.5  0.75 1.  ]


In [ ]:
# SIMULATION + FINAL STATE TRACKING
t0 = time.perf_counter()
ineff_df = {}
final_states_dict = {}

for eta_minus in eta_minus_list:
    key = fmt_eta(float(eta_minus))
    mean_expect, final_states = run_homodyne_avg_eta_minus_with_finals(
        rho0=rho0,
        times=times,
        gamma=float(gamma),
        phi1=float(phi1),
        phi2=float(phi2),
        eta_minus=float(eta_minus),
        e_ops=e_ops,
        ntraj=int(ntraj),
        chunk_size=int(chunk_size),
        num_cpus=int(num_cpus),
        seed=int(seed + 1000 * float(eta_minus)),
    )

    df = pd.DataFrame(mean_expect.T, columns=columns)
    df.insert(0, "step", np.arange(len(times), dtype=int))
    ineff_df[key] = df
    final_states_dict[key] = final_states

runtime_sim = float(time.perf_counter() - t0)
print(f"Simulation completed in {runtime_sim:.2f} s")
for eta_minus in eta_minus_list:
    key = fmt_eta(float(eta_minus))
    print(f"eta_minus={key}: n_final_states={len(final_states_dict[key])}")


10.0%. Run time:  12.16s. Est. time left: 00:00:01:49
20.0%. Run time:  12.24s. Est. time left: 00:00:00:48
30.0%. Run time:  12.27s. Est. time left: 00:00:00:28
40.0%. Run time:  12.48s. Est. time left: 00:00:00:18
50.0%. Run time:  12.61s. Est. time left: 00:00:00:12
60.0%. Run time:  12.64s. Est. time left: 00:00:00:08
70.0%. Run time:  14.36s. Est. time left: 00:00:00:06
80.0%. Run time:  14.38s. Est. time left: 00:00:00:03
90.0%. Run time:  14.41s. Est. time left: 00:00:00:01
100.0%. Run time:  14.47s. Est. time left: 00:00:00:00
Total run time:  15.24s
[eta_minus=0.000] chunk 1: 10/30 trajectories completed


In [ ]:
# SAVE CURVES + FINAL STATES + RUN CONFIG
phi1_dir = angle_to_path(float(phi1))
phi2_dir = angle_to_path(float(phi2))
base_name = f"state={state_kind}_phi_1={phi1_dir}_phi_2={phi2_dir}"
if tag:
    base_name = f"{base_name}_{tag}"
out_dir = out_root / base_name
out_dir.mkdir(parents=True, exist_ok=True)

meta = {
    "measurement": "homodyne",
    "N": 2,
    "state": state_kind,
    "theta": float(theta),
    "phi_state": float(phi_state),
    "gamma": float(gamma),
    "phi1": float(phi1),
    "phi2": float(phi2),
    "eta_plus_fixed": float(eta_plus),
    "eta_minus_list": [float(x) for x in eta_minus_list],
    "ntraj": int(ntraj),
    "chunk_size": int(chunk_size),
    "dt_T1": float(dt),
    "t_end_T1": float(t_end),
    "num_cpus": int(num_cpus),
    "seed": int(seed),
    "columns": columns,
    "runtime_sec": runtime_sim,
    "out_dir": str(out_dir),
}

npz_path = out_dir / f"homodyne_eta_minus_scan_phi1={float(phi1):.6g}_phi2={float(phi2):.6g}.npz"
save_ineff_df_npz(ineff_df, npz_path, meta=None, step_col="step")

qsave(final_states_dict, out_dir / "final_states_dict")
(out_dir / "run_config.json").write_text(json.dumps(meta, indent=2), encoding="utf-8")

print(f"Saved in: {out_dir}")


In [ ]:
# PLOT OBSERVABLES (Conc and Xi2_KU)
phi1_tex = angle_to_tex(float(phi1))
phi2_tex = angle_to_tex(float(phi2))
x = times / T1

for col in columns:
    plt.figure(figsize=(12, 8))
    for eta_minus in eta_minus_list:
        key = fmt_eta(float(eta_minus))
        y = ineff_df[key][col].to_numpy()
        plt.plot(x, y, label=rf"$\eta_- = {eta_minus:.2f}$")

    plt.xlim(0.0, float(t_end))
    plt.xlabel(r"$t/T_1$")
    plt.ylabel(LABELS[col])
    plt.title(
        r"Homodyne $J_z$ (N=2), $\eta_+=1$: "
        + LABELS[col]
        + rf"$,\ \phi_1={phi1_tex}\ \phi_2={phi2_tex}$"
        + rf"  (n$_\mathrm{{traj}}$={ntraj})"
    )
    plt.grid(True, linestyle=":", alpha=0.6)
    plt.legend(loc="upper left", bbox_to_anchor=(0, 0.95), fontsize=12)
    plt.savefig(out_dir / f"Homodyne_eta_minus_scan_{col}.pdf", bbox_inches="tight")
    plt.show()


In [ ]:
# FINAL-STATE FIDELITY TABLES + HISTOGRAMS (no best_state ranking)
library = build_candidate_library()
state_labels = list(library.keys())
avg_state_fidelities = {}

for eta_minus in eta_minus_list:
    key = fmt_eta(float(eta_minus))

    ftab = fidelity_table(final_states_dict[key], library)
    ftab.to_csv(out_dir / f"final_state_fidelities_eta_minus={key}.csv", index=False)

    plot_fidelity_histograms(
        ftab=ftab,
        state_labels=state_labels,
        eta_minus=float(eta_minus),
        out_path=out_dir / f"final_state_fidelity_hist_eta_minus={key}.pdf",
    )

    rho_avg = average_final_state(final_states_dict[key])
    if rho_avg is not None:
        avg_state_fidelities[key] = {
            lab: float(np.real_if_close(fidelity(rho_avg, st))) for lab, st in library.items()
        }
    else:
        avg_state_fidelities[key] = {}

(out_dir / "avg_state_fidelities.json").write_text(
    json.dumps(avg_state_fidelities, indent=2),
    encoding="utf-8",
)

print("Fidelity analysis completed.")
print(f"Output folder: {out_dir}")
